In [2]:
from pathlib import Path
import pandas as pd
import numpy as np

DATA_DIR = Path("../data/raw")

train_df = pd.read_csv(
    DATA_DIR / "UNSW_NB15_training-set.csv"
)

test_df = pd.read_csv(
    DATA_DIR / "UNSW_NB15_testing-set.csv"
)

# Separate features (X) from the binary target (y)
excluded_columns = ["id", "label", "attack_cat"]

X_train = train_df.drop(columns=excluded_columns)
y_train = train_df["label"]

X_test = test_df.drop(columns=excluded_columns)
y_test = test_df["label"]

print("X_train:", X_train.shape)
print("y_train:", y_train.shape)

print("X_test:", X_test.shape)
print("y_test:", y_test.shape)

print("\nTarget distribution:")
print(y_train.value_counts())

X_train: (175341, 42)
y_train: (175341,)
X_test: (82332, 42)
y_test: (82332,)

Target distribution:
label
1    119341
0     56000
Name: count, dtype: int64


In [3]:
categorical_columns = X_train.select_dtypes(
    include=["object", "str", "category"]
).columns.tolist()

numerical_columns = X_train.select_dtypes(
    include=np.number
).columns.tolist()

print("Categorical columns:", categorical_columns)
print("Number of categorical columns:", len(categorical_columns))

print("\nNumber of numerical columns:", len(numerical_columns))
print(numerical_columns)

Categorical columns: ['proto', 'service', 'state']
Number of categorical columns: 3

Number of numerical columns: 39
['dur', 'spkts', 'dpkts', 'sbytes', 'dbytes', 'rate', 'sttl', 'dttl', 'sload', 'dload', 'sloss', 'dloss', 'sinpkt', 'dinpkt', 'sjit', 'djit', 'swin', 'stcpb', 'dtcpb', 'dwin', 'tcprtt', 'synack', 'ackdat', 'smean', 'dmean', 'trans_depth', 'response_body_len', 'ct_srv_src', 'ct_state_ttl', 'ct_dst_ltm', 'ct_src_dport_ltm', 'ct_dst_sport_ltm', 'ct_dst_src_ltm', 'is_ftp_login', 'ct_ftp_cmd', 'ct_flw_http_mthd', 'ct_src_ltm', 'ct_srv_dst', 'is_sm_ips_ports']


In [4]:
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.pipeline import Pipeline

numeric_pipeline = Pipeline([
    ("scaler", StandardScaler())
])

categorical_pipeline = Pipeline([
    (
        "encoder",
        OneHotEncoder(handle_unknown="ignore")
    )
])

preprocessor = ColumnTransformer([
    ("num", numeric_pipeline, numerical_columns),
    ("cat", categorical_pipeline, categorical_columns)
])

# Learn preprocessing parameters from training data only
X_train_processed = preprocessor.fit_transform(X_train)

# Apply the same learned transformations to test data
X_test_processed = preprocessor.transform(X_test)

print("Processed training shape:", X_train_processed.shape)
print("Processed testing shape:", X_test_processed.shape)

Processed training shape: (175341, 194)
Processed testing shape: (82332, 194)


In [5]:
print("Training rows preserved:",
      X_train_processed.shape[0] == len(y_train))

print("Testing rows preserved:",
      X_test_processed.shape[0] == len(y_test))

print("Training target values:",
      sorted(y_train.unique()))

print("Processed feature count:",
      X_train_processed.shape[1])


Training rows preserved: True
Testing rows preserved: True
Training target values: [np.int64(0), np.int64(1)]
Processed feature count: 194


In [6]:
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.pipeline import Pipeline

def build_preprocessor(numerical_columns, categorical_columns):
    numeric_pipeline = Pipeline([
        ("scaler", StandardScaler())
    ])

    categorical_pipeline = Pipeline([
        ("encoder", OneHotEncoder(handle_unknown="ignore"))
    ])

    return ColumnTransformer([
        ("num", numeric_pipeline, numerical_columns),
        ("cat", categorical_pipeline, categorical_columns)
    ])

preprocessor = build_preprocessor(
    numerical_columns,
    categorical_columns
)

# Learn transformations from training data only
X_train_processed = preprocessor.fit_transform(X_train)

# Reuse those transformations for test data
X_test_processed = preprocessor.transform(X_test)

print("Training shape:", X_train_processed.shape)
print("Testing shape:", X_test_processed.shape)
print("Feature names learned:", len(preprocessor.get_feature_names_out()))


Training shape: (175341, 194)
Testing shape: (82332, 194)
Feature names learned: 194


In [7]:

def prepare_input(df):
    df = df.copy()
    df.columns = df.columns.str.strip()

    # Remove identifiers and labels if present
    df = df.drop(
        columns=["id", "label", "attack_cat"],
        errors="ignore"
    )

    expected_columns = X_train.columns.tolist()

    missing_columns = [
        col for col in expected_columns
        if col not in df.columns
    ]

    if missing_columns:
        raise ValueError(
            f"Missing required features: {missing_columns}"
        )

    # Select the exact feature set and order used during training
    return df[expected_columns]


# Test the function using the held-out test data
sample_input = prepare_input(test_df.head(5))

sample_processed = preprocessor.transform(sample_input)

print("Input shape:", sample_input.shape)
print("Processed shape:", sample_processed.shape)


Input shape: (5, 42)
Processed shape: (5, 194)


In [8]:

print("Training rows preserved:",
      X_train_processed.shape[0] == len(y_train))

print("Testing rows preserved:",
      X_test_processed.shape[0] == len(y_test))

print("Feature dimensions match:",
      X_train_processed.shape[1] == X_test_processed.shape[1])

print("Uploaded CSV transformation works:",
      sample_processed.shape == (5, X_train_processed.shape[1]))


Training rows preserved: True
Testing rows preserved: True
Feature dimensions match: True
Uploaded CSV transformation works: True
